In [1]:
"""
Borough name-matching check between gapscore_v2.csv and boundary geojson.
Run this BEFORE loading anything into Tableau to catch mismatches early.
"""

import pandas as pd
import geopandas as gpd

# ---- 1. Load your data ----
gap = pd.read_csv(r"C:\Users\Hp\Downloads\Project 2026 DS\ActiveLives_Data\gapscore_v2.csv")

# Adjust path to wherever you save the downloaded boundary file
gdf = gpd.read_file(r"C:\Users\Hp\Downloads\Project 2026 DS\boundaryfile.geojson")

# ---- 2. Your 33 London boroughs (adjust if your existing list differs) ----
london_boroughs = [
    "Barking and Dagenham", "Barnet", "Bexley", "Brent", "Bromley", "Camden",
    "Croydon", "Ealing", "Enfield", "Greenwich", "Hackney",
    "Hammersmith and Fulham", "Haringey", "Harrow", "Havering", "Hillingdon",
    "Hounslow", "Islington", "Kensington and Chelsea", "Kingston upon Thames",
    "Lambeth", "Lewisham", "Merton", "Newham", "Redbridge",
    "Richmond upon Thames", "Southwark", "Sutton", "Tower Hamlets",
    "Waltham Forest", "Wandsworth", "Westminster", "City of London"
]

# ---- 3. Find the name column in the geojson (usually LAD24NM or similar) ----
print("GeoDataFrame columns:", gdf.columns.tolist())
name_col = "LAD24NM"  # change this if your file uses a different column name

# ---- 4. Filter geojson down to London boroughs only ----
gdf_london = gdf[gdf[name_col].isin(london_boroughs)].copy()
print(f"\nMatched {len(gdf_london)} / 33 boroughs by direct name match in geojson.")

# ---- 5. Cross-check sets ----
gap_names = set(gap['borough'].unique())
geo_names = set(gdf_london[name_col].unique())
expected = set(london_boroughs)

print("\n--- In gapscore_v2.csv but NOT in filtered geojson ---")
print(gap_names - geo_names or "None — all matched")

print("\n--- In filtered geojson but NOT in gapscore_v2.csv ---")
print(geo_names - gap_names or "None — all matched")

print("\n--- Expected 33 boroughs missing from geojson entirely ---")
print(expected - set(gdf[name_col].unique()) or "None — all 33 present in raw geojson")

print("\n--- Expected 33 boroughs missing from gapscore_v2.csv ---")
print(expected - gap_names or "None — all 33 present in gapscore")

# ---- 6. If mismatches found, inspect raw geojson names for near-matches ----
if gap_names - geo_names:
    print("\n--- Checking raw geojson for near-matches (case/spacing/hyphen issues) ---")
    all_geo_names = gdf[name_col].unique()
    for missing in (gap_names - geo_names):
        close = [n for n in all_geo_names if missing.lower().replace("-", " ").strip()
                  in n.lower().replace("-", " ").strip()
                  or n.lower().replace("-", " ").strip()
                  in missing.lower().replace("-", " ").strip()]
        print(f"  '{missing}' -> possible matches in geojson: {close}")

# ---- 7. Save the filtered, clean London-only boundary file ----
if len(gdf_london) == 33 and not (gap_names - geo_names) and not (geo_names - gap_names):
    out_path = r"C:\Users\Hp\Downloads\Project 2026 DS\GAP_data\london_boundaries_clean.geojson"
    gdf_london.to_file(out_path, driver="GeoJSON")
    print(f"\nAll names matched cleanly. Saved filtered file to:\n{out_path}")
else:
    print("\nMismatches found above — fix names in gapscore_v2.csv or apply a rename "
          "dict before saving the filtered geojson. Do NOT load into Tableau yet.")

GeoDataFrame columns: ['FID', 'LAD24CD', 'LAD24NM', 'LAD24NMW', 'BNG_E', 'BNG_N', 'LONG', 'LAT', 'GlobalID', 'geometry']

Matched 33 / 33 boroughs by direct name match in geojson.

--- In gapscore_v2.csv but NOT in filtered geojson ---
None — all matched

--- In filtered geojson but NOT in gapscore_v2.csv ---
None — all matched

--- Expected 33 boroughs missing from geojson entirely ---
None — all 33 present in raw geojson

--- Expected 33 boroughs missing from gapscore_v2.csv ---
None — all 33 present in gapscore

All names matched cleanly. Saved filtered file to:
C:\Users\Hp\Downloads\Project 2026 DS\GAP_data\london_boundaries_clean.geojson


In [2]:
import os

files_to_check = [
    r"C:\Users\Hp\Downloads\Project 2026 DS\ActiveLives_Data\gapscore_v2.csv",
    r"C:\Users\Hp\Downloads\Project 2026 DS\GAP_data\gap_scores.csv",
    r"C:\Users\Hp\Downloads\Project 2026 DS\GAP_data\gap_scores_full33.csv",
    r"C:\Users\Hp\Downloads\Project 2026 DS\GAP_data\gap_scores_equity.csv",
]

for f in files_to_check:
    if os.path.exists(f):
        mtime = os.path.getmtime(f)
        import datetime
        print(f"{f}\n  last modified: {datetime.datetime.fromtimestamp(mtime)}\n")
    else:
        print(f"{f}\n  NOT FOUND\n")

C:\Users\Hp\Downloads\Project 2026 DS\ActiveLives_Data\gapscore_v2.csv
  last modified: 2026-07-31 17:55:32.873831

C:\Users\Hp\Downloads\Project 2026 DS\GAP_data\gap_scores.csv
  last modified: 2026-07-31 18:10:42.986645

C:\Users\Hp\Downloads\Project 2026 DS\GAP_data\gap_scores_full33.csv
  last modified: 2026-07-31 18:10:42.986645

C:\Users\Hp\Downloads\Project 2026 DS\GAP_data\gap_scores_equity.csv
  last modified: 2026-08-02 01:47:11.098106



In [3]:
import pandas as pd
df = pd.read_csv(r"C:\Users\Hp\Downloads\Project 2026 DS\GAP_data\gap_scores_equity.csv")
print(df.columns.tolist())
print(df.shape)
print(df[['borough']].nunique())  # should be 33
print(df.isnull().sum())  # check for unexpected gaps, especially in classification/equity_gap columns
df.head(10)

['borough', 'inactive', 'readiness_opportunity', 'readiness_ability', 'respondents', 'sessions', 'venues', 'readiness_opportunity_pct', 'inactive_rank', 'readiness_opp_rank', 'demand_score', 's_bin', 'd_bin', 'class', 's_bin2', 'd_bin2', 'class_median', 's_bin4', 'd_bin4', 'class_quartile', 'concern', 'concern_median', 'concern_quartile', 'stable', 'demand_score_shrunk', 'd_bin_shrunk', 'note', 'deprived_inactive', 'deprived_sample', 'equity_gap', 'equity_flag']
(32, 31)
borough    32
dtype: int64
borough                       0
inactive                      0
readiness_opportunity         0
readiness_ability             0
respondents                   0
sessions                      0
venues                        0
readiness_opportunity_pct     0
inactive_rank                 0
readiness_opp_rank            0
demand_score                  0
s_bin                         0
d_bin                         0
class                         0
s_bin2                        0
d_bin2           

,borough,inactive,readiness_opportunity,readiness_ability,respondents,sessions,venues,readiness_opportunity_pct,inactive_rank,readiness_opp_rank,...,concern_median,concern_quartile,stable,demand_score_shrunk,d_bin_shrunk,note,deprived_inactive,deprived_sample,equity_gap,equity_flag
0,Barking and Dagenham,37.653268,3.831661,4.090338,2913.0,2535.0,3.0,70.791533,100.000,3.125,...,low,low,True,51.685142,mid,"genuine puzzle, not a data issue - real, recen...",40.719339,3441.0,3.066072,low gap
1,Barnet,25.524509,3.937450,4.092044,2944.0,639.0,6.0,73.436240,56.250,43.750,...,low,low,True,50.839244,mid,NaN,34.782973,664.0,9.258464,high gap
2,Bexley,27.855703,3.844326,4.005352,2464.0,475.0,16.0,71.108157,71.875,6.250,...,low,low,True,45.489690,low,NaN,38.041909,647.0,10.186206,high gap
3,Brent,31.181177,3.926900,4.088565,2536.0,2587.0,11.0,73.172488,93.750,37.500,...,mid,mid,True,58.777949,high,NaN,33.442637,1586.0,2.261460,low gap
4,Bromley,22.184714,4.029240,4.134204,2449.0,105.0,2.0,75.730990,43.750,87.500,...,high,high,True,58.657546,high,NaN,30.276599,433.0,8.091885,high gap
5,Camden,18.438633,3.944899,4.099771,2497.0,159.0,7.0,73.622480,12.500,50.000,...,low,low,True,41.542155,low,NaN,21.962388,1384.0,3.523755,low gap
6,Croydon,27.126883,3.907103,4.114872,2494.0,65.0,5.0,72.677574,65.625,31.250,...,low,mid,False,50.134343,mid,NaN,30.931169,1238.0,3.804286,low gap
7,Ealing,27.293569,3.851310,4.119147,2509.0,3427.0,11.0,71.282742,68.750,15.625,...,low,low,True,46.997708,low,NaN,29.043675,1311.0,1.750107,low gap
8,Enfield,28.378248,3.859744,4.057659,2984.0,77.0,6.0,71.493597,75.000,21.875,...,low,mid,False,49.982688,low,reclassified from genuine desert - real centre...,35.034820,1602.0,6.656572,high gap
9,Greenwich,25.176496,3.954611,4.145419,3455.0,137.0,7.0,73.865278,53.125,56.250,...,high,mid,False,53.487511,high,NaN,35.817851,1887.0,10.641355,high gap
